In [1]:
import hail as hl
import os
from ukb_utils import hail_init
from ukb_utils import genotypes
from ukb_utils import variants

In [2]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('logs/hail/hail_test_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe091.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to logs/hail/hail_test_export.log


In [3]:
mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt')
#mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt')

#mt = genotypes.get_ukb_imputed_v3_bgen([20])
#mt = variants.liftover(mt)
#mt = mt.rows()

In [4]:
mt = hl.vep(mt, 'utils/configs/vep_dbNSFP.json')


In [5]:
mt.vep

<StructExpression of type struct{assembly_name: str, allele_string: str, ancestral: str, colocated_variants: array<struct{aa_allele: str, aa_maf: float64, afr_allele: str, afr_maf: float64, allele_string: str, amr_allele: str, amr_maf: float64, clin_sig: array<str>, end: int32, eas_allele: str, eas_maf: float64, ea_allele: str, ea_maf: float64, eur_allele: str, eur_maf: float64, exac_adj_allele: str, exac_adj_maf: float64, exac_allele: str, exac_afr_allele: str, exac_afr_maf: float64, exac_amr_allele: str, exac_amr_maf: float64, exac_eas_allele: str, exac_eas_maf: float64, exac_fin_allele: str, exac_fin_maf: float64, exac_maf: float64, exac_nfe_allele: str, exac_nfe_maf: float64, exac_oth_allele: str, exac_oth_maf: float64, exac_sas_allele: str, exac_sas_maf: float64, id: str, minor_allele: str, minor_allele_freq: float64, phenotype_or_disease: int32, pubmed: array<int32>, sas_allele: str, sas_maf: float64, somatic: int32, start: int32, strand: int32}>, context: str, end: int32, id: st

In [7]:
mt.vep.transcript_consequences.provean_pred.show()

,,
locus,alleles,
locus<GRCh38>,array<str>,array<float64>
chr21:10413617,"[""C"",""T""]","[NA,NA]"
chr21:10413618,"[""G"",""A""]","[NA,NA]"
chr21:10413629,"[""C"",""T""]","[NA,NA]"
chr21:10413634,"[""T"",""G""]","[NA,NA]"
chr21:10413636,"[""G"",""T""]","[NA,NA]"
chr21:10413653,"[""A"",""G""]","[NA,NA]"
chr21:10413662,"[""C"",""T""]","[NA,NA]"
chr21:10413708,"[""TGGAGCAGTAAGATGGCGGCC"",""T""]","[NA,NA]"


In [7]:
#mt = hl.read_matrix_table('data/mt/ukb_wes_200k_annotated_chr21.mt')
#hl.read_table('data/qc_new/ukb_wes_200k_chr15_variants_phased.ht').show()

In [121]:
rsids = hl.literal('')
for chr in range(21,22):
    mfi  =genotypes.get_ukb_imputed_v3_mfi(chr)
    mfi = mfi.filter((mfi.maf < 0.01) & 
                     (mfi.maf > 0.0001) &
                     (mfi.info == 1))
    rsid = mfi.rsid
    rsids += rsid
rsids = hl.set(rsids.collect())

2021-10-19 11:14:13 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)
  Loading field 'f2' as type int32 (user-supplied)
  Loading field 'f3' as type str (not specified)
  Loading field 'f4' as type str (not specified)
  Loading field 'f5' as type float64 (user-supplied)
  Loading field 'f6' as type str (not specified)
  Loading field 'f7' as type float64 (user-supplied)
2021-10-19 11:14:14 Hail: INFO: Ordering unsorted dataset with network shuffle


In [122]:
mt2 = genotypes.get_ukb_imputed_v3_bgen([21,22], entry_fields = ['GT'])
mt2.filter_rows(rsids.contains(mt2.rsid))

2021-10-19 11:14:21 Hail: INFO: Number of BGEN files parsed: 2
2021-10-19 11:14:21 Hail: INFO: Number of samples in BGEN files: 487409
2021-10-19 11:14:21 Hail: INFO: Number of variants across all BGEN files: 2516841


In [124]:
mt = genotypes.get_ukb_imputed_v3_bgen([21], entry_fields = ['GT'])

2021-10-19 11:24:20 Hail: INFO: Number of BGEN files parsed: 1
2021-10-19 11:24:20 Hail: INFO: Number of samples in BGEN files: 487409
2021-10-19 11:24:20 Hail: INFO: Number of variants across all BGEN files: 1261158


In [130]:
hl.set(mt.s.collect()).contains(mt.s)

<BooleanExpression of type bool>